Backend

In [1]:
from fastapi import FastAPI
from pydantic import BaseModel
from fastapi.middleware.cors import CORSMiddleware
import requests

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],      # For development only
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class SearchRequest(BaseModel):
    startdate: str
    enddate: str
    stocklist: str
    top: int = 20
    market: str = "nasdaq"
        


from fastapi.responses import HTMLResponse
from fastapi.templating import Jinja2Templates
from fastapi import Request

templates = Jinja2Templates(directory="templates")
@app.get("/", response_class=HTMLResponse)
def dashboard(request: Request):

    return templates.TemplateResponse(
        "open interest.html",
        {
            "request": request
        }
    )


@app.post("/search")
def search(request: SearchRequest):
    
    print("startdate =", request.startdate, type(request.startdate))
    print("enddate   =", request.enddate, type(request.enddate))
    print("stocklist =", request.stocklist)
    
    try:
        df = run_option_dashboard_query(
        startdate=request.startdate,
        enddate=request.enddate,
        stocklist=request.stocklist,
        market=request.market,
        top=request.top
    )
        dashboard = build_dashboard_data(df)
        
    except RuntimeError as e:
        print(e)
        dashboard = build_dashboard_data(df)
        
    return dashboard

In [2]:
# ==========================================================
# Complete Query Pipeline
# ==========================================================

def run_option_dashboard_query(
    startdate,
    enddate,
    stocklist,
    market="nasdaq",
    top=20
):

    # ==========================================================
    # 1. Query option turnover API
    # ==========================================================

    result = query_option_turnover(
        startdate=startdate,
        enddate=enddate,
        stocklist=stocklist,
        market=market,
        top=top
    )


    # ==========================================================
    # 2. Convert JSON to DataFrame
    # ==========================================================

    raw_df = option_dataframe(result)


    if raw_df.empty:
        return pd.DataFrame()


    # ==========================================================
    # 3. Prepare DataFrame
    # ==========================================================

    df = prepare_dashboard_dataframe(raw_df)


    rename = {

        "option_code": "Ticker",

        "formatted_expiry": "Expiry",

        "call_type": "Type",

        "strike": "Strike",

        "vol": "Volume",

        "open_int": "Open Interest",

        "iv": "IV",

        "delta": "Delta",

        "gamma": "Gamma",

        "theta": "Theta",

        "vega": "Vega",

        "underlying_price_1545": "Underlying Price"

    }


    df.rename(
        columns=rename,
        inplace=True
    )


    # ==========================================================
    # 4. Prepare dates
    # ==========================================================

    today = pd.Timestamp.today().normalize()


    df["Expiry"] = pd.to_datetime(
        df["Expiry"],
        format="%Y%m%d"
    )


    df["DTE"] = (
        df["Expiry"] - today
    ).dt.days


    # Target month used by the OI API
    df["TargetMonth"] = (
        df["Expiry"]
        .dt.strftime("%Y%m")
    )


    # ==========================================================
    # 5. Query OI summary API
    #
    # One API call per unique
    # stock + target month combination
    # ==========================================================

    oi_frames = []


    unique_combinations = (
        df[
            ["Ticker", "TargetMonth"]
        ]
        .drop_duplicates()
    )


    for _, row in unique_combinations.iterrows():

        stockcode = str(row["Ticker"])

        targetmonth = str(row["TargetMonth"])


        try:

            oi_result = query_open_interest_summary(
                targetmonth=targetmonth,
                stockcode=stockcode,
                market=market,
                changetype="per"
            )


            oi_temp = open_interest_dataframe(
                oi_result
            )


            if not oi_temp.empty:

                oi_temp["TargetMonth"] = targetmonth

                oi_frames.append(
                    oi_temp
                )


        except Exception as e:

            print(
                f"OI API failed: "
                f"{market} / "
                f"{stockcode} / "
                f"{targetmonth}: "
                f"{e}"
            )


    # ==========================================================
    # 6. Merge OI data
    # ==========================================================

    if oi_frames:

        oi_df = pd.concat(
            oi_frames,
            ignore_index=True
        )


        # Make sure ticker has the same type
        # on both sides of the merge

        df["Ticker"] = (
            df["Ticker"]
            .astype(str)
        )


        oi_df["stock_code"] = (
            oi_df["stock_code"]
            .astype(str)
        )


        # Merge by:
        #
        # Ticker
        # TargetMonth
        # Type
        # Strike

        df = df.merge(

            oi_df[
                [
                    "stock_code",
                    "TargetMonth",
                    "Type",
                    "Strike",
                    "OI_API"
                ]
            ],

            left_on=[
                "Ticker",
                "TargetMonth",
                "Type",
                "Strike"
            ],

            right_on=[
                "stock_code",
                "TargetMonth",
                "Type",
                "Strike"
            ],

            how="left"
        )


        # Remove duplicate ticker column
        df.drop(
            columns=["stock_code"],
            inplace=True,
            errors="ignore"
        )


    else:

        # If OI API returned nothing,
        # still provide the column.

        df["OI_API"] = pd.NA


    # ==========================================================
    # 7. Additional calculations
    # ==========================================================

    if ("Underlying Price" in df.columns and "Strike" in df.columns):
        df["Moneyness"] = (df["Underlying Price"]/df["Strike"])
    else:
        df["Moneyness"] = pd.NA

    if "Underlying Price" in df.columns:
        df["Status"] = df.apply(option_status, axis=1)
    else:
        df["Status"] = pd.NA

    df["Vol/OI"] = (
        df["Volume"]
        /
        df["Open Interest"].replace(0, pd.NA)
    ).round(2)


    # ==========================================================
    # 8. Sort
    # ==========================================================

    df = df.sort_values(
        "Open Interest",
        ascending=False
    )


    # ==========================================================
    # 9. Remove unnecessary columns
    # ==========================================================

    drop_columns = [

        "stock_code",

        "expiry",

        "qdate",

        "formatted_qdate",

        "size",

        "mvdiff",

        "oidiff"

    ]


    df.drop(
        columns=drop_columns,
        inplace=True,
        errors="ignore"
    )


    # ==========================================================
    # 10. Select display columns
    # ==========================================================

    display_columns = [

        "Ticker",

        "Expiry",

        "DTE",

        "Type",

        "Strike",

        "Underlying Price",

        "Status",

        "bid",

        "ask",

        "last",

        "chg",

        "Volume",

        "Open Interest",

        "OI_API",

        "Vol/OI",

        "IV",

        "Delta"

    ]


    # Only select columns that actually exist
    display_columns = [
        col
        for col in display_columns
        if col in df.columns
    ]


    df = df[display_columns]


    # ==========================================================
    # 11. Format date
    # ==========================================================

    df["Expiry"] = (
        df["Expiry"]
        .dt.strftime("%Y-%m-%d")
    )

    return df

In [3]:
# ==========================================================
# Option Turnover API
# ==========================================================

def query_option_turnover(
    startdate,
    enddate,
    stocklist,
    market="nasdaq",
    top=20,
    threshold=0,
    source="db",
    changetype="per",
    thresholdtype="all"
):
    """
    Query option turnover API.

    Returns
    -------
    dict
        Raw JSON returned by the API.
    """

    params = {

        "q":"optionturnoverquery",

        "source":source,

        "startdate":startdate,

        "enddate":enddate,

        "stocklist":stocklist,

        "market":market,

        "top":top,

        "threshold":threshold,

        "changetype":changetype,

        "thresholdtype":thresholdtype,

        "compress":0,

        "format":"json",

        "secret": "54309e61afcec341aaf3636f1f5f57b0a537b9ce",

        "lang":"en"

    }
    
    BASE_URL = "https://bithkex.com/api/"
    
    HEADERS = {"User-Agent": "Mozilla/5.0"}

    response = requests.get(
        BASE_URL,
        params=params,
        headers=HEADERS,
        timeout=30
    )

    response.raise_for_status()

    data = response.json()

    if data.get("code") != 0:
        
        print(data.get("msg"))
        
        return {"code": 0, "res": []}

    return data

In [4]:
import pandas as pd

# ==========================================================
# Open Interest Summary API
# ==========================================================
def query_open_interest_summary(
    targetmonth,
    stockcode,
    market="hkex",
    changetype="per"
):
    """
    Query summarized open interest for a stock and target month.

    Parameters
    ----------
    targetmonth : str
        Contract month, e.g. "202608".

    stockcode : str
        Stock code, e.g. "00388".

    market : str
        Market, e.g. "hkex".

    changetype : str
        Change type requested by the API.

    Returns
    -------
    dict
        Raw JSON returned by the API.
    """

    params = {

        "q": "oisumquery",

        "sum": "open_int",

        "targetmonth": targetmonth,

        "stockcode": stockcode,

        "market": market,

        "changetype": changetype,

        "compress": 0,

        "format": "json",

        "secret": "54309e61afcec341aaf3636f1f5f57b0a537b9ce",

        "lang": "en"
    }
    
    BASE_URL = "https://bithkex.com/api/"
    
    HEADERS = {"User-Agent": "Mozilla/5.0"}

    response = requests.get(
        BASE_URL,
        params=params,
        headers=HEADERS,
        timeout=30
    )

    response.raise_for_status()

    data = response.json()

    if data.get("code") != 0:
        raise RuntimeError(
            f"OI API Error: {data.get('msg')}"
        )

    return data

# ==========================================================
# Convert OI Summary API response to DataFrame
# ==========================================================

def open_interest_dataframe(api_result):

    if "res" not in api_result:
        return pd.DataFrame()

    df = pd.DataFrame(api_result["res"])

    # Rename API fields
    df.rename(
        columns={
            "stock_code": "stock_code",
            "call_type": "Type",
            "strike": "Strike",
            "sum(open_int)": "OI_API"
        },
        inplace=True
    )

    # Make sure numeric fields are numeric
    if "Strike" in df.columns:
        df["Strike"] = pd.to_numeric(
            df["Strike"],
            errors="coerce"
        )

    if "OI_API" in df.columns:
        df["OI_API"] = pd.to_numeric(
            df["OI_API"],
            errors="coerce"
        )

    return df


In [5]:
oi_result = query_open_interest_summary(
    targetmonth="202608",
    stockcode="00388",
    market="hkex",
    changetype="per"
)

oi_df = open_interest_dataframe(oi_result)

display(oi_df)

,stock_code,Type,Strike,OI_API
0,00388,P,200,0
1,00388,C,200,59
2,00388,P,205,0
3,00388,C,205,59
4,00388,P,210,0
...,...,...,...,...
163,00388,C,690,0
164,00388,P,700,0
165,00388,C,700,0
166,00388,P,710,0


In [6]:
# ==========================================================
# Convert API response to DataFrame
# ==========================================================

def option_dataframe(api_result):

    if "res" not in api_result:

        return pd.DataFrame()

    df = pd.DataFrame(api_result["res"])

    return df

In [7]:
# ==========================================================
# Prepare dashboard data
# ==========================================================

def prepare_dashboard_dataframe(df):

    df = df.copy()

    rename = {

        "option_code":"Ticker",

        "formatted_expiry":"Expiry",

        "call_type":"Type",

        "strike":"Strike",

        "vol":"Volume",

        "open_int":"Open Interest",

        "iv":"IV",

        "delta":"Delta",

        "gamma":"Gamma",

        "theta":"Theta",

        "vega":"Vega"

    }

    for old, new in rename.items():

        if old in df.columns:

            df.rename(columns={old:new}, inplace=True)

    numeric = [

        "Strike",

        "Volume",

        "Open Interest",

        "IV",

        "Delta",

        "Gamma",

        "Theta",

        "Vega"

    ]

    for col in numeric:

        if col in df.columns:

            df[col] = pd.to_numeric(df[col], errors="coerce")
            
    df = df.drop(columns=["implied_underlying_price_1545"], errors="ignore")

    return df

In [8]:
# ==========================================================
# Dashboard Summary
# ==========================================================

def dashboard_summary(df):

    if len(df) == 0:

        return {

            "tickerCount":0,

            "contractCount":0,

            "maxOI":0,

            "avgIV":0

        }

    summary = {

        "tickerCount":

            df["Ticker"].nunique(),

        "contractCount":

            len(df),

        "maxOI":

            int(df["Open Interest"].max()),

        "avgIV":

            round(df["IV"].mean(),4)

    }

    return summary

In [9]:
# ==========================================================
# Prepare dashboard object
# ==========================================================

def dashboard_data(df):

    summary = dashboard_summary(df)

    rows = df.to_dict(orient="records")

    return {

        "summary":summary,

        "rows":rows

    }

In [10]:
def option_status(row):

    if row["Type"]=="C":

        if row["Underlying Price"] > row["Strike"]:

            return "ITM"

        elif abs(row["Underlying Price"]-row["Strike"])<1:

            return "ATM"

        else:

            return "OTM"

    else:

        if row["Underlying Price"] < row["Strike"]:

            return "ITM"

        elif abs(row["Underlying Price"]-row["Strike"])<1:

            return "ATM"

        else:

            return "OTM"

In [11]:
# ==========================================================
# Build Dashboard Data
# ==========================================================

def build_dashboard_data(df):

    summary = {

        "tickerCount": int(df["Ticker"].nunique()),

        "contractCount": int(len(df)),

        "maxOI": int(df["Open Interest"].max()),

        "avgIV": round(float(df["IV"].mean()),3)

    }

    data = {

        "summary": summary,
        
        "columns": df.columns.tolist(),

        "rows": df.to_dict(orient="records")

    }

    return data

In [12]:
import threading
import uvicorn
import pandas as pd

def run_server():
    uvicorn.run(
        app,
        host="127.0.0.1",
        port=8000,
        log_level="info"
    )

server_thread = threading.Thread(
    target=run_server,
    daemon=True
)

server_thread.start()

In [13]:
test_df = run_option_dashboard_query(
    startdate="20260627",
    enddate="20260726",
    stocklist="AAPL,MSFT",
    market="nasdaq",
    top=20
)

display(test_df.head())

INFO:     Started server process [12500]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
C:\Users\25280\AppData\Local\Temp\ipykernel_12500\2346804529.py:198: UserWarning: You are merging on int and float columns where the float values are not equal to their int representation.
  df = df.merge(


,Ticker,Expiry,DTE,Type,Strike,Underlying Price,Status,bid,ask,last,chg,Volume,Open Interest,OI_API,Vol/OI,IV,Delta
0,AAPL,2026-10-16,67,C,360,332.280,OTM,8.75,9.00,9.20,1.80,2590,75299,80138,0.03,0.2681,0.3272
4,AAPL,2026-08-21,11,C,345,332.280,OTM,6.95,7.10,7.40,2.25,4184,41114,35598,0.10,0.2815,0.3674
1,AAPL,2026-09-18,39,C,285,327.770,ITM,46.95,47.70,47.25,2.59,10142,25591,6245,0.40,0.3349,0.8733
15,AAPL,2026-09-18,39,C,300,321.115,ITM,28.05,29.55,29.06,-1.24,503,25546,28482,0.02,0.2989,0.7525
7,AAPL,2026-07-20,-21,P,335,328.045,ITM,6.85,7.10,8.40,4.80,22333,22579,6940,0.99,1.0718,-0.9826


In [14]:
test_df = run_option_dashboard_query(
    startdate="20260801",
    enddate="20260810",
    stocklist="00388",
    market="hkex",
    top=20
)

display(test_df.head(20))

OI API failed: hkex / HEX / 202608: OI API Error: No result. 无结果
OI API failed: hkex / HEX / 202611: OI API Error: No result. 无结果
OI API failed: hkex / HEX / 202703: OI API Error: No result. 无结果
OI API failed: hkex / HEX / 202706: OI API Error: No result. 无结果
OI API failed: hkex / HEX / 202609: OI API Error: No result. 无结果
OI API failed: hkex / HEX / 202610: OI API Error: No result. 无结果


KeyError: 'Volume'

In [ ]:
print(
    test_df[
        [
            "Ticker",
            "Expiry",
            "Type",
            "Strike",
            "Open Interest",
            "OI_API"
        ]
    ].head(20)
)

Old version, static dashboard (Abandoned)

Create html

In [2]:
import requests
import pandas as pd
import numpy as np
import json
from datetime import datetime
from pathlib import Path

# ==========================================================
# Configuration
# ==========================================================

SECRET = "61f2fd18f1f0634195d6fd07a147f8af870289b4"

BASE_URL = "https://bithkex.com/api/"

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

In [3]:
# ==========================================================
# Option Turnover API
# ==========================================================

def query_option_turnover(
    startdate,
    enddate,
    stocklist,
    market="nasdaq",
    top=20,
    threshold=0,
    source="db",
    changetype="per",
    thresholdtype="all"
):
    """
    Query option turnover API.

    Returns
    -------
    dict
        Raw JSON returned by the API.
    """

    params = {

        "q":"optionturnoverquery",

        "source":source,

        "startdate":startdate,

        "enddate":enddate,

        "stocklist":stocklist,

        "market":market,

        "top":top,

        "threshold":threshold,

        "changetype":changetype,

        "thresholdtype":thresholdtype,

        "compress":0,

        "format":"json",

        "secret": "61f2fd18f1f0634195d6fd07a147f8af870289b4",

        "lang":"en"

    }
    
    BASE_URL = "https://bithkex.com/api/"
    
    HEADERS = {"User-Agent": "Mozilla/5.0"}

    response = requests.get(
        BASE_URL,
        params=params,
        headers=HEADERS,
        timeout=30
    )

    response.raise_for_status()

    data = response.json()

    if data.get("code") != 0:
        raise RuntimeError(
            f"API Error: {data.get('msg')}"
        )

    return data

In [4]:
# ==========================================================
# Convert API response to DataFrame
# ==========================================================

def option_dataframe(api_result):

    if "res" not in api_result:

        return pd.DataFrame()

    df = pd.DataFrame(api_result["res"])

    return df

In [5]:
# ==========================================================
# Prepare dashboard data
# ==========================================================

def prepare_dashboard_dataframe(df):

    df = df.copy()

    rename = {

        "option_code":"Ticker",

        "formatted_expiry":"Expiry",

        "call_type":"Type",

        "strike":"Strike",

        "vol":"Volume",

        "open_int":"Open Interest",

        "iv":"IV",

        "delta":"Delta",

        "gamma":"Gamma",

        "theta":"Theta",

        "vega":"Vega"

    }

    for old, new in rename.items():

        if old in df.columns:

            df.rename(columns={old:new}, inplace=True)

    numeric = [

        "Strike",

        "Volume",

        "Open Interest",

        "IV",

        "Delta",

        "Gamma",

        "Theta",

        "Vega"

    ]

    for col in numeric:

        if col in df.columns:

            df[col] = pd.to_numeric(df[col], errors="coerce")
            
    df = df.drop(columns=["implied_underlying_price_1545"], errors="ignore")

    return df

In [6]:
# ==========================================================
# Dashboard Summary
# ==========================================================

def dashboard_summary(df):

    if len(df) == 0:

        return {

            "tickerCount":0,

            "contractCount":0,

            "maxOI":0,

            "avgIV":0

        }

    summary = {

        "tickerCount":

            df["Ticker"].nunique(),

        "contractCount":

            len(df),

        "maxOI":

            int(df["Open Interest"].max()),

        "avgIV":

            round(df["IV"].mean(),4)

    }

    return summary

In [7]:
# ==========================================================
# Prepare dashboard object
# ==========================================================

def dashboard_data(df):

    summary = dashboard_summary(df)

    rows = df.to_dict(orient="records")

    return {

        "summary":summary,

        "rows":rows

    }

In [8]:
# Query API
result = query_option_turnover(
    startdate="20260627",
    enddate="20260726",
    stocklist="AAPL,MSFT",
    market="nasdaq",
    top=20
)

# Convert to DataFrame
raw_df = option_dataframe(result)

# Prepare dashboard DataFrame
dashboard_df = prepare_dashboard_dataframe(raw_df)

# Build dashboard object
dashboard = dashboard_data(dashboard_df)

print(dashboard["summary"])

display(dashboard_df.head())

{'tickerCount': 2, 'contractCount': 20, 'maxOI': 75299, 'avgIV': 0.6017}


,Ticker,stock_code,expiry,Type,Strike,qdate,Open Interest,last,chg,bid,...,Theta,Vega,rho,vwap,open,high,low,underlying_price_1545,formatted_qdate,Expiry
0,AAPL,AAPL,2026-10-15T16:00:00.000Z,C,360,2026-07-15T16:00:00.000Z,75299,9.20,1.80,8.75,...,-0.0856,0.6037,0.2560,8.1925,7.40,9.85,6.85,332.280,20260716,20261016
1,AAPL,AAPL,2026-09-17T16:00:00.000Z,C,285,2026-07-20T16:00:00.000Z,25591,47.25,2.59,46.95,...,-0.0710,0.2759,0.3993,46.3575,44.66,48.85,42.05,327.770,20260721,20260918
2,AAPL,AAPL,2026-09-17T16:00:00.000Z,C,285,2026-07-21T16:00:00.000Z,15556,44.00,-1.24,43.60,...,-0.0715,0.2815,0.3896,44.3080,45.24,45.24,43.30,324.445,20260722,20260918
3,AAPL,AAPL,2026-09-17T16:00:00.000Z,C,285,2026-07-22T16:00:00.000Z,6050,41.20,0.25,40.85,...,-0.0708,0.3036,0.3722,40.8695,40.95,41.54,40.19,321.115,20260723,20260918
4,AAPL,AAPL,2026-08-20T16:00:00.000Z,C,345,2026-07-15T16:00:00.000Z,41114,7.40,2.25,6.95,...,-0.1521,0.3935,0.1193,6.4856,5.15,8.10,4.80,332.280,20260716,20260821


In [9]:
rename = {

    "option_code":"Ticker",

    "formatted_expiry":"Expiry",

    "call_type":"Type",

    "strike":"Strike",

    "vol":"Volume",

    "open_int":"Open Interest",

    "iv":"IV",

    "delta":"Delta",

    "gamma":"Gamma",

    "theta":"Theta",

    "vega":"Vega",

    "underlying_price_1545":"Underlying Price"

}


dashboard_df.rename(columns=rename, inplace=True)


Additional information and cleaning

In [10]:
def option_status(row):

    if row["Type"]=="C":

        if row["Underlying Price"] > row["Strike"]:

            return "ITM"

        elif abs(row["Underlying Price"]-row["Strike"])<1:

            return "ATM"

        else:

            return "OTM"

    else:

        if row["Underlying Price"] < row["Strike"]:

            return "ITM"

        elif abs(row["Underlying Price"]-row["Strike"])<1:

            return "ATM"

        else:

            return "OTM"

In [11]:
today = pd.Timestamp.today().normalize()

dashboard_df["Expiry"] = pd.to_datetime(dashboard_df["Expiry"], format="%Y%m%d")

dashboard_df["DTE"] = (dashboard_df["Expiry"] - today).dt.days

dashboard_df["Moneyness"] = (dashboard_df["Underlying Price"] / dashboard_df["Strike"])

dashboard_df["Status"] = dashboard_df.apply(option_status, axis=1)

dashboard_df["Vol/OI"] = (dashboard_df["Volume"] / dashboard_df["Open Interest"]).round(2)

dashboard_df = dashboard_df.sort_values("Open Interest", ascending=False)

#dashboard_df = dashboard_df.sort_values("Volume", ascending=False)

In [12]:
dashboard_df.head()

,Ticker,stock_code,expiry,Type,Strike,qdate,Open Interest,last,chg,bid,...,open,high,low,Underlying Price,formatted_qdate,Expiry,DTE,Moneyness,Status,Vol/OI
0,AAPL,AAPL,2026-10-15T16:00:00.000Z,C,360,2026-07-15T16:00:00.000Z,75299,9.20,1.80,8.75,...,7.40,9.85,6.85,332.280,20260716,2026-10-16,79,0.923000,OTM,0.03
4,AAPL,AAPL,2026-08-20T16:00:00.000Z,C,345,2026-07-15T16:00:00.000Z,41114,7.40,2.25,6.95,...,5.15,8.10,4.80,332.280,20260716,2026-08-21,23,0.963130,OTM,0.10
1,AAPL,AAPL,2026-09-17T16:00:00.000Z,C,285,2026-07-20T16:00:00.000Z,25591,47.25,2.59,46.95,...,44.66,48.85,42.05,327.770,20260721,2026-09-18,51,1.150070,ITM,0.40
15,AAPL,AAPL,2026-09-17T16:00:00.000Z,C,300,2026-07-22T16:00:00.000Z,25546,29.06,-1.24,28.05,...,30.30,31.00,27.85,321.115,20260723,2026-09-18,51,1.070383,ITM,0.02
7,AAPL,AAPL,2026-07-19T16:00:00.000Z,P,335,2026-07-19T16:00:00.000Z,22579,8.40,4.80,6.85,...,3.60,10.70,2.83,328.045,20260720,2026-07-20,-9,0.979239,ITM,0.99


In [13]:
print(dashboard_df.columns.tolist())

['Ticker', 'stock_code', 'expiry', 'Type', 'Strike', 'qdate', 'Open Interest', 'last', 'chg', 'bid', 'ask', 'Volume', 'size', 'mvdiff', 'oidiff', 'IV', 'Delta', 'Gamma', 'Theta', 'Vega', 'rho', 'vwap', 'open', 'high', 'low', 'Underlying Price', 'formatted_qdate', 'Expiry', 'DTE', 'Moneyness', 'Status', 'Vol/OI']


In [14]:
# Remove duplicates

drop_columns = [
    "stock_code",          # same as Ticker
    "expiry",              # duplicate of Expiry
    "qdate",
    "formatted_qdate",
    "size",                # always 100
    "mvdiff",
    "oidiff"
]

dashboard_df.drop(
    columns=drop_columns,
    inplace=True,
    errors="ignore"
)

In [15]:
# What information to display

display_columns = [
    "Ticker",
    "Expiry",
    "DTE",
    "Type",
    "Strike",
    "Underlying Price",
    "Status",
    "bid",
    "ask",
    "last",
    "chg",
    "Volume",
    "Open Interest",
    "Vol/OI",
    "IV",
    "Delta"
]

dashboard_display_df = dashboard_df[display_columns].copy()

dashboard_display_df = dashboard_display_df.copy()

dashboard_display_df["Expiry"] = dashboard_display_df["Expiry"].dt.strftime("%Y-%m-%d")

Build html

In [16]:
# ==========================================================
# Build Dashboard Data
# ==========================================================

def build_dashboard_data(df):

    summary = {

        "tickerCount": int(df["Ticker"].nunique()),

        "contractCount": int(len(df)),

        "maxOI": int(df["Open Interest"].max()),

        "avgIV": round(float(df["IV"].mean()),3)

    }

    data = {

        "summary": summary,
        
        "columns": df.columns.tolist(),

        "rows": df.to_dict(orient="records")

    }

    return data

In [18]:
import json
import webbrowser
from pathlib import Path


def generate_dashboard(
    df,
    template="open interest.html",
    output="open interest dashboard.html"
):

    dashboard = build_dashboard_data(df)

    dashboard_json = json.dumps(
        dashboard,
        default=str,
        indent=2
    )
    
    print(dashboard_json)

    with open(template, "r", encoding="utf-8") as f:
        html = f.read()

    html = html.replace(
        "{{OPTION_DATA}}",
        dashboard_json
    )
    

    with open(output, "w", encoding="utf-8") as f:
        f.write(html)

    print(f"Dashboard saved to {output}")

In [20]:
generate_dashboard(dashboard_display_df)

{
  "summary": {
    "tickerCount": 2,
    "contractCount": 20,
    "maxOI": 75299,
    "avgIV": 0.602
  },
  "columns": [
    "Ticker",
    "Expiry",
    "DTE",
    "Type",
    "Strike",
    "Underlying Price",
    "Status",
    "bid",
    "ask",
    "last",
    "chg",
    "Volume",
    "Open Interest",
    "Vol/OI",
    "IV",
    "Delta"
  ],
  "rows": [
    {
      "Ticker": "AAPL",
      "Expiry": "2026-10-16",
      "DTE": 79,
      "Type": "C",
      "Strike": 360,
      "Underlying Price": 332.28,
      "Status": "OTM",
      "bid": 8.75,
      "ask": 9.0,
      "last": 9.2,
      "chg": 1.8,
      "Volume": 2590,
      "Open Interest": 75299,
      "Vol/OI": 0.03,
      "IV": 0.2681,
      "Delta": 0.3272
    },
    {
      "Ticker": "AAPL",
      "Expiry": "2026-08-21",
      "DTE": 23,
      "Type": "C",
      "Strike": 345,
      "Underlying Price": 332.28,
      "Status": "OTM",
      "bid": 6.95,
      "ask": 7.1,
      "last": 7.4,
      "chg": 2.25,
      "Volume": 4184,

FileNotFoundError: [Errno 2] No such file or directory: 'open interest.html'

In [ ]:
dashboard = build_dashboard_data(dashboard_display_df)

print(dashboard.keys())